# Tunisia Solar Priority Zones

**Authors:** Seifeddine Abroud & Ghezlen Ben Abdrahmen, Institut Supérieur d’Informatique de Mahdia

**Research question:** Can machine learning identify screening zones for solar energy development in Tunisia by combining solar potential, population, land cover, and infrastructure access?

**Expectations to test**
1. Southern cells will generally have high solar resource and low population density.
2. Northern/coastal cells will generally be closer to roads and settlements, with more built-up and cropland cover.
3. Clustering will expose trade-offs rather than declare any location suitable for construction.

> This is a screening/learning exercise, not a siting decision. Run top-to-bottom in Google Colab. Outputs are saved to Drive so they survive runtime resets. Solar Atlas rasters and Earth Engine authorization are optional manual prerequisites; missing layers are never fabricated.

## Goals and work in progress — 12 October 2026

**Goals**
- Build an equal-area 10 km grid clipped to Tunisia and attach governorate labels.
- Aggregate real source layers, explore them, compare K-Means solutions, and map cluster profiles.

**Work in progress**
- This notebook is the project scaffold and analysis pipeline. Boundary and WorldPop layers are downloaded from their published sources; Global Solar Atlas files need to be downloaded/uploaded by the researcher. Earth Engine, OSM, and NASA POWER stages are optional.
- Keep source dates, units, and missing-data decisions documented below as you run the workflow.

## Fix log (review pass)
- **Removed the synthetic-data sections (old 12-16).** They generated PVOUT, DNI, land cover and distances from latitude, so the clusters and the "top 15 sites" were circular. Real layers only from now on.
- Added a one-time cleanup cell that quarantines any outputs produced from the synthetic data.
- Clustering now **stops** unless a real solar layer is present.
- Replaced the hand-weighted ranking with a **cluster-based shortlist** whose ranking is tested over 1,000 random weightings.
- Fixed: MultiPolygon/GeometryCollection cells (Earth Engine stage), solar file auto-detection, population total sanity check, correlated-feature warning, `K_OVERRIDE`, unused code.
- Use **Runtime > Restart and run all** before trusting any output.

## Model-comparison pass
- **Section 10** now compares K-Means with five other clustering models (hierarchical, probabilistic, tree-based, graph-based, density-based) using three internal scores, agreement (ARI) and the overlap of the high-solar zone.
- **Section 10b** adds Gaussian Mixture BIC and a per-cell confidence map; **Section 10c** adds supervised checks (decision tree, random forest, permutation importance) with spatial cross-validation.
- **Section 12b** tests whether the shortlist survives a change of model and prints a summary.
- Fixed: false 'outside rough range' alarm on yearly solar files, non-existent cluster numbers in `CANDIDATE_CLUSTERS` silently ignored, hard-coded 'Top 15'/'Top 60' titles.


## 0. Setup and Drive persistence

Run in Colab. Mount Drive and set `BASE` to the existing project folder. The package install can take a few minutes.

In [ ]:
%pip -q install geopandas rasterio rasterstats osmnx earthengine-api geemap scikit-learn
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/tunisia_solar')
for p in ['raw/boundaries','raw/population','raw/solar','processed','figures']:
    (BASE/p).mkdir(parents=True, exist_ok=True)
print('Project folder:', BASE)

In [ ]:
import os, json, warnings, requests, numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from shapely.geometry import box, mapping, Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union
import rasterio
from IPython.display import display
from rasterstats import zonal_stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
RNG=42
CRS_METRIC='EPSG:32632'
plt.rcParams['figure.dpi']=120
pd.set_option('display.max_columns', 100)
model_df=None        # created in Section 7 only when real data are sufficient
CLUSTER_DONE=False   # set True in Section 9


### One-time cleanup of the earlier synthetic outputs
The previous version of this notebook wrote *fabricated* solar, land-cover and infrastructure tables using the same file names the real data will use. This cell moves them (and the figures made from them) to `processed/_quarantine_synthetic/` so they cannot be merged by accident. It runs once, then leaves a marker file. You can delete the quarantine folder afterwards.

In [ ]:
Q=BASE/'processed'/'_quarantine_synthetic'; MARK=BASE/'processed'/'_cleanup_done.txt'
if not MARK.exists():
    Q.mkdir(parents=True, exist_ok=True)
    moved=[]
    for rel in ['processed/grid_solar.csv','processed/grid_landcover.csv','processed/grid_infrastructure.csv',
                'processed/features_clean.csv','processed/cluster_profiles.csv','processed/clustered_cells.csv',
                'figures/clusters_map.png','figures/PVOUT_map.png','figures/top_solar_zones_chart.png']:
        f=BASE/rel
        if f.exists(): f.rename(Q/f.name); moved.append(f.name)
    MARK.write_text('synthetic outputs quarantined')
    print('Moved to', Q, '->', moved)
else:
    print('Cleanup already done.')


## 1. Tunisia boundary, governorates, and equal-area grid

Fetch official GeoJSON metadata via geoBoundaries. Grid cells are squares in UTM metres (10 km default); each cell is clipped to the country boundary, so edge cells have smaller actual area. We preserve true geometry and use actual area for population density.

In [ ]:
def geoboundaries(adm):
    url=f'https://www.geoboundaries.org/api/current/gbOpen/TUN/{adm}/'
    r=requests.get(url, timeout=60); r.raise_for_status(); meta=r.json()
    print(adm, 'metadata:', {k:meta.get(k) for k in ['boundaryName','boundaryISO','boundaryYear','gjDownloadURL']})
    gj=requests.get(meta['gjDownloadURL'], timeout=120); gj.raise_for_status()
    out=BASE/'raw'/'boundaries'/f'TUN_{adm}.geojson'; out.write_bytes(gj.content)
    return gpd.read_file(out), meta

country, meta0 = geoboundaries('ADM0')
governorates, meta1 = geoboundaries('ADM1')
country=country.to_crs(CRS_METRIC); governorates=governorates.to_crs(CRS_METRIC)
# GeoBoundaries property names can vary; inspect to choose the human-readable governorate field.
print('ADM1 fields:', governorates.columns.tolist())
country[['geometry']].plot(figsize=(6,7), color='#dceef4', edgecolor='#243746'); plt.title('Tunisia boundary'); plt.axis('equal'); plt.show()

In [ ]:
# Choose likely name field automatically; override NAME_COL if inspection shows another field.
name_candidates=[c for c in governorates.columns if c.lower() in ('shapeName'.lower(),'name','name_1','admin','shape_name')]
NAME_COL=name_candidates[0] if name_candidates else next(c for c in governorates.columns if c!='geometry')
print('Governorate label field:', NAME_COL)
CELL_KM=10
geom=country.geometry.union_all()
minx,miny,maxx,maxy=geom.bounds
step=CELL_KM*1000
xs=np.arange(np.floor(minx/step)*step, maxx, step); ys=np.arange(np.floor(miny/step)*step, maxy, step)
raw=[box(x,y,x+step,y+step) for x in xs for y in ys if box(x,y,x+step,y+step).intersects(geom)]
grid=gpd.GeoDataFrame({'cell_id':[f'TUN_{i:05d}' for i in range(len(raw))]}, geometry=raw, crs=CRS_METRIC)

def polys_only(g):
    # intersection() can return GeometryCollections (lines/points at edges); keep polygon parts only
    if g.is_empty or isinstance(g,(Polygon,MultiPolygon)): return g
    if isinstance(g,GeometryCollection):
        parts=[p for p in g.geoms if isinstance(p,(Polygon,MultiPolygon))]
        return unary_union(parts) if parts else Polygon()
    return Polygon()

grid['geometry']=grid.geometry.intersection(geom).apply(polys_only)
grid=grid[~grid.geometry.is_empty].copy()
grid['area_km2']=grid.geometry.area/1e6
MIN_AREA_KM2=10   # drop sliver cells (<10% of a full 100 km2 cell): their density values are noisy
n_before=len(grid); grid=grid[grid.area_km2>=MIN_AREA_KM2].reset_index(drop=True)
print(f'Dropped {n_before-len(grid)} sliver cells under {MIN_AREA_KM2} km2')
cent=grid.geometry.centroid
grid['lon']=gpd.GeoSeries(cent,crs=CRS_METRIC).to_crs(4326).x
grid['lat']=gpd.GeoSeries(cent,crs=CRS_METRIC).to_crs(4326).y
# One label per cell: use largest-overlap governorate where a cell crosses a boundary.
over=grid[['cell_id','geometry']].merge(governorates[[NAME_COL,'geometry']], how='cross')
over['overlap']=over.apply(lambda r:r.geometry_x.intersection(r.geometry_y).area,axis=1)
labels=over.sort_values('overlap').drop_duplicates('cell_id',keep='last').set_index('cell_id')[NAME_COL]
grid['governorate']=grid.cell_id.map(labels)
print('Cells:',len(grid),'Governorates:',grid.governorate.nunique())
grid.to_file(BASE/'processed'/'grid_geometry.geojson', driver='GeoJSON')
grid.drop(columns='geometry').to_csv(BASE/'processed'/'grid_geometry.csv',index=False)
ax=governorates.boundary.plot(figsize=(8,8),color='#40566b',linewidth=.6)
grid.boundary.plot(ax=ax,color='#8ca0ae',linewidth=.15); country.boundary.plot(ax=ax,color='#111827',linewidth=1.1)
ax.set_title(f'Tunisia equal-area grid ({CELL_KM} km nominal cells)'); ax.set_axis_off(); plt.show()

## 2. Solar resource rasters (Global Solar Atlas)

Download the Tunisia GIS GeoTIFF package from the Global Solar Atlas in a browser; read its metadata and place files in `raw/solar/`. Use one consistent temporal convention (daily averages recommended). The cell below **auto-detects** each layer (searching `raw/solar/` recursively) and prefers the *LTAy_AvgDailyTotals* version; it prints every candidate it found so you can verify the choice. Do not include GHI, GTI, and PVOUT simultaneously in K-Means without a deliberate weighting rationale.

In [ ]:
SOLAR_DIR=BASE/'raw'/'solar'
LAYERS=['GHI','DNI','PVOUT','OPTA','TEMP']

def find_layer(layer):
    cands=[p for p in SOLAR_DIR.rglob('*') if p.suffix.lower() in ('.tif','.tiff') and layer.lower() in p.stem.lower()]
    cands=[p for p in cands if 'monthly' not in str(p).lower()]          # skip monthly layers
    daily=[p for p in cands if 'avgdaily' in str(p).lower().replace('_','')]
    pool=daily or cands
    if len(pool)>1: print(f'{layer}: several candidates, using the shortest path ->', [p.name for p in pool])
    return min(pool,key=lambda p:len(str(p))) if pool else None

SOLAR_FILES={l:find_layer(l) for l in LAYERS}
print({k:(v.name if v else None) for k,v in SOLAR_FILES.items()})

def add_raster_zonal(frame, layer, path, stat='mean'):
    if not path or not Path(path).exists():
        print(f'SKIP {layer}: no GeoTIFF found in {SOLAR_DIR}'); return frame
    with rasterio.open(path) as src:
        print(layer, '| file:', Path(path).name, '| CRS:', src.crs, '| nodata:', src.nodata, '| description:', src.descriptions)
        g=frame.to_crs(src.crs)
        vals=zonal_stats(g.geometry, str(path), stats=[stat], nodata=src.nodata, all_touched=False)
    frame[layer]=[v.get(stat) for v in vals]
    return frame

for layer,path in SOLAR_FILES.items(): grid=add_raster_zonal(grid,layer,path,'mean')
solar_cols=[c for c in LAYERS if c in grid]

# Sanity check against rough expected ranges for Tunisia (given as DAILY averages; scaled x365 when the file holds yearly totals)
EXPECTED={'GHI':(3.5,7.5),'DNI':(3.0,8.5),'PVOUT':(3.0,6.0),'OPTA':(20,45)}
for l,(lo,hi) in EXPECTED.items():
    if l in grid and grid[l].notna().any():
        mn,mx=grid[l].min(),grid[l].max()
        yearly=l!='OPTA' and grid[l].median()>100      # yearly totals (kWh per year) are ~365x the daily averages
        if yearly: lo,hi=lo*365,hi*365
        print(f'{l}: {mn:.2f} to {mx:.2f}', '(yearly totals)' if yearly else '(daily averages)' if l!='OPTA' else '', '' if (lo<=mn and mx<=hi) else '  <-- outside rough range: check units or nodata')

if solar_cols:
    grid[['cell_id']+solar_cols].to_csv(BASE/'processed'/'grid_solar.csv',index=False)
    if 'PVOUT' in grid and grid.PVOUT.notna().any():
        grid.plot(column='PVOUT',legend=True,cmap='magma',figsize=(7,8),missing_kwds={'color':'lightgrey'}); plt.title('Mean PVOUT by grid cell'); plt.axis('off'); plt.show()
else:
    print('No solar layers loaded yet. Download the Global Solar Atlas Tunisia GeoTIFFs and put them in', SOLAR_DIR)


## 3. Population (WorldPop 2020)

Population is a people-per-pixel count: **sum** it within cells, then divide by actual cell area in km². This uses the official 2020 Tunisia raster URL. The optional constrained raster may be smaller; if the file name or URL has changed, verify it from the WorldPop TUN directory before proceeding.

In [ ]:
import urllib.request
POP_URL='https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/TUN/tun_ppp_2020.tif'
POP_PATH=BASE/'raw'/'population'/'tun_ppp_2020.tif'
if not POP_PATH.exists():
    try:
        urllib.request.urlretrieve(POP_URL, POP_PATH)
        print('Downloaded:', POP_PATH, 'MB=',round(POP_PATH.stat().st_size/1e6,1))
    except Exception as e: print('Download failed; verify current WorldPop TUN filename/URL:',repr(e))
if POP_PATH.exists():
    with rasterio.open(POP_PATH) as src:
        print('Population raster:',src.crs,src.nodata,src.width,src.height)
        g=grid.to_crs(src.crs)
        zs=zonal_stats(g.geometry, str(POP_PATH), stats=['sum'], nodata=src.nodata, all_touched=False)
    grid['population']=[z.get('sum') or 0 for z in zs]
    grid['pop_density']=grid.population/grid.area_km2
    print(f'Total population in grid: {grid.population.sum():,.0f}  (Tunisia is roughly 11-12 million; a big gap means a clipping or nodata problem)')
    grid[['cell_id','population','pop_density']].to_csv(BASE/'processed'/'grid_population.csv',index=False)
    grid.plot(column='pop_density',legend=True,cmap='viridis',figsize=(7,8),norm=None); plt.title('WorldPop 2020 population density (people/km²)'); plt.axis('off'); plt.show()

## 4. Land cover (ESA WorldCover 2021 v200; optional Earth Engine)

This section requires an Earth Engine account/project and Colab authentication. It samples land-cover class proportions for each grid polygon in batches. ESA WorldCover class codes: 10 tree, 20 shrub, 30 grass, 40 cropland, 50 built-up, 60 bare/sparse, 80 water. Cells with no valid class pixels may remain missing.

In [ ]:
RUN_EE=True  # set True after Earth Engine access is approved and configured
EE_PROJECT='your-earth-engine-project'   # replace with your own Google Earth Engine project ID
if RUN_EE:
    import ee
    try: ee.Initialize(project=EE_PROJECT)
    except Exception: ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
    image=ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
    # Use a compact feature collection; each polygon is reduced at 10 m in manageable batches.
    ee_gdf=grid.to_crs(4326)
    feats=[]
    for start in range(0,len(ee_gdf),250):
        batch=ee_gdf.iloc[start:start+250]
        # mapping() handles Polygon and MultiPolygon (e.g. cells touching Djerba/Kerkennah) and keeps holes
        ee_features=[ee.Feature(ee.Geometry(mapping(row.geometry),None,False),{'cell_id':row.cell_id}) for _,row in batch.iterrows()]
        fc=ee.FeatureCollection(ee_features)
        hist=image.reduceRegions(collection=fc,reducer=ee.Reducer.frequencyHistogram(),scale=10,tileScale=4)
        feats.extend(hist.getInfo()['features'])
        print('Processed cells:', min(start+250,len(ee_gdf)))
    # Convert class histograms to percent composition.
    class_groups={'pct_builtup':[50],'pct_cropland':[40],'pct_bare':[60], 'pct_water':[80], 'pct_tree_shrub_grass':[10,20,30]}
    rows=[]
    for f in feats:
        h=next((v for v in f['properties'].values() if isinstance(v,dict)),{}); h={str(int(float(k))):v for k,v in h.items()}; total=sum(float(v) for v in h.values())
        d={'cell_id':f['properties']['cell_id']}
        for name,codes in class_groups.items(): d[name]=100*sum(float(h.get(str(c),0)) for c in codes)/total if total else np.nan
        rows.append(d)
    lc=pd.DataFrame(rows); lc.to_csv(BASE/'processed'/'grid_landcover.csv',index=False)
    grid=grid.merge(lc,on='cell_id',how='left')
else: print('Earth Engine stage skipped; set RUN_EE=True after setup.')

## 5. Infrastructure distances from OpenStreetMap (optional)

OSM power mapping is incomplete in some areas; absence of mapped infrastructure is not proof of absence. OSMnx/Overpass extraction can be rate-limited or large. Run only when network access is reliable; keep these distances as descriptive proxies, not grid capacity.

In [ ]:
RUN_OSM=True
if RUN_OSM:
    import requests
    OSM_DIR=BASE/'raw'/'osm'; OSM_DIR.mkdir(parents=True,exist_ok=True)
    PBF_URL='https://download.geofabrik.de/africa/tunisia-latest.osm.pbf'   # ~80 MB, whole country, one file
    CACHE={k:OSM_DIR/f'tn_{k}.gpkg' for k in ['roads','power_lines','substations']}

    if not all(p.exists() for p in CACHE.values()):
        PBF=Path('/content/tunisia-latest.osm.pbf')       # local Colab disk is faster than Drive
        if not PBF.exists():
            print('Downloading the Tunisia OpenStreetMap extract from Geofabrik (~80 MB)...')
            with requests.get(PBF_URL,stream=True,timeout=120) as r:
                r.raise_for_status()
                with open(PBF,'wb') as fh:
                    for chunk in r.iter_content(1<<20): fh.write(chunk)
        print('Downloaded:',round(PBF.stat().st_size/1e6),'MB. Reading lines (roads + power lines)... this takes a few minutes')

        POWER_LINE_RE='"power"=>"line"|"power"=>"minor_line"'
        lines_all=gpd.read_file(PBF,layer='lines',engine='pyogrio',
            where="highway IN ('motorway','trunk','primary') OR other_tags LIKE '%\"power\"=>\"line\"%' OR other_tags LIKE '%\"power\"=>\"minor_line\"%'")
        roads=lines_all[lines_all['highway'].isin(['motorway','trunk','primary'])]
        plines=lines_all[lines_all['other_tags'].fillna('').str.contains(POWER_LINE_RE,regex=True)]

        print('Reading substations / power plants...')
        SUB_WHERE="other_tags LIKE '%\"power\"=>\"substation\"%' OR other_tags LIKE '%\"power\"=>\"plant\"%'"
        sub_pts=gpd.read_file(PBF,layer='points',engine='pyogrio',where=SUB_WHERE)
        sub_pol=gpd.read_file(PBF,layer='multipolygons',engine='pyogrio',where=SUB_WHERE)
        subs=pd.concat([sub_pts[['geometry']],sub_pol[['geometry']]],ignore_index=True)

        print(f'Found: {len(roads)} major road segments | {len(plines)} power-line segments | {len(subs)} substations/plants')
        assert len(roads)>0 and len(plines)>0 and len(subs)>0, 'One of the layers came back empty: send me the line above.'
        roads[['geometry']].to_file(CACHE['roads'],driver='GPKG')
        plines[['geometry']].to_file(CACHE['power_lines'],driver='GPKG')
        gpd.GeoDataFrame(subs,geometry='geometry',crs=4326).to_file(CACHE['substations'],driver='GPKG')
    else:
        print('Using cached OSM layers from Drive.')

    roads=gpd.read_file(CACHE['roads']).to_crs(CRS_METRIC)
    lines=gpd.read_file(CACHE['power_lines']).to_crs(CRS_METRIC)
    subs=gpd.read_file(CACHE['substations']).to_crs(CRS_METRIC)
    pts=gpd.GeoDataFrame({'cell_id':grid.cell_id},geometry=grid.geometry.centroid,crs=CRS_METRIC)

    def nearest_dist(points, features):
        j=gpd.sjoin_nearest(points[['geometry']],features[['geometry']],how='left',distance_col='d')
        return (j[~j.index.duplicated()]['d']/1000).reindex(points.index).to_numpy()

    grid['dist_road_km']=nearest_dist(pts,roads)
    grid['dist_power_line_km']=nearest_dist(pts,lines)
    grid['dist_substation_km']=nearest_dist(pts,subs)
    grid[['cell_id','dist_road_km','dist_power_line_km','dist_substation_km']].to_csv(BASE/'processed'/'grid_infrastructure.csv',index=False)
    print(grid[['dist_road_km','dist_power_line_km','dist_substation_km']].describe().round(1))
else: print('OSM stage skipped; set RUN_OSM=True.')

## 6. Optional climate (NASA POWER)

NASA POWER is coarse; use only as contextual climate descriptors, not as a replacement for Global Solar Atlas. This optional example requests a small sample near central Tunisia. For a full project, query unique ~0.5° blocks and cache results; do not make thousands of duplicate calls.

In [ ]:
RUN_POWER=False
if RUN_POWER:
    params='T2M,WS10M,RH2M,PRECTOTCORR'
    def nasa_power(lat,lon):
        u='https://power.larc.nasa.gov/api/temporal/climatology/point'
        j=requests.get(u,params={'parameters':params,'community':'RE','longitude':lon,'latitude':lat,'format':'JSON'},timeout=60).json()
        return {k:v.get('ANN') for k,v in j['properties']['parameter'].items()}
    sample=nasa_power(35.0,9.0); print(sample)
else: print('NASA POWER stage skipped (optional).')

## 7. Merge, inspect, transform, and save clean features

Join available processed tables by `cell_id`; report missingness. Do not silently impute missing primary source layers. Choose a feature set after inspecting correlation; the default is intentionally conservative. Population and distances use `log1p`; then standardize for clustering.

In [ ]:
model_df=None
# Merge previously saved per-phase tables (REAL data only) into the grid table.
for fname in ['grid_solar.csv','grid_population.csv','grid_landcover.csv','grid_infrastructure.csv']:
    p=BASE/'processed'/fname
    if p.exists():
        extra=pd.read_csv(p)
        cols=[c for c in extra if c=='cell_id' or c not in grid.columns]
        grid=grid.merge(extra[cols],on='cell_id',how='left')

STATUS={'solar':any(c in grid for c in ['GHI','DNI','PVOUT']),'population':'pop_density' in grid,
        'land cover':'pct_builtup' in grid,'roads/power':'dist_power_line_km' in grid}
print('Real layers loaded:',STATUS)
missing=grid.drop(columns='geometry',errors='ignore').isna().mean().sort_values(ascending=False)
print('Missing fraction by field:\n',missing.to_string())

num=grid.select_dtypes(include='number').columns.tolist()
if num:
    grid[num].hist(figsize=(14,10),bins=30); plt.tight_layout(); plt.show()
    plt.figure(figsize=(10,8)); sns.heatmap(grid[num].corr(),cmap='coolwarm',center=0); plt.title('Feature correlation'); plt.show()

# One solar variable by default: GHI, GTI, DNI and PVOUT are strongly correlated and would give solar extra weight.
# Add 'DNI' only after reading the heatmap and deciding that it adds information.
CANDIDATES=['PVOUT','pop_density','pct_builtup','pct_cropland','pct_bare','dist_road_km','dist_power_line_km']
FEATURES=[c for c in CANDIDATES if c in grid and grid[c].notna().any()]
if 'pop_density' in FEATURES: grid['log_pop_density']=np.log1p(grid['pop_density']); FEATURES[FEATURES.index('pop_density')]='log_pop_density'
for c in ['dist_road_km','dist_power_line_km']:
    if c in FEATURES: grid['log_'+c]=np.log1p(grid[c].clip(lower=0)); FEATURES[FEATURES.index(c)]='log_'+c
print('Selected features:',FEATURES)

SOLAR_IN=[c for c in FEATURES if c in ('PVOUT','DNI','GHI','GTI')]
if not SOLAR_IN:
    print('STOP: no real solar layer yet, so clustering is skipped. Run Section 2 with the Global Solar Atlas GeoTIFFs.')
elif len(FEATURES)<3:
    print('STOP: need at least 3 measured features (solar + population + one more). Run the missing sections first.')
else:
    corr=grid[FEATURES].corr().abs()
    pairs=[(a,b,round(float(corr.loc[a,b]),2)) for i,a in enumerate(FEATURES) for b in FEATURES[i+1:] if corr.loc[a,b]>0.85]
    if pairs: print('WARNING highly correlated feature pairs (consider dropping one):',pairs)
    model_df=grid.dropna(subset=FEATURES).copy()
    print('Complete-case cells:',len(model_df),'of',len(grid),'dropped:',len(grid)-len(model_df))
    model_df.drop(columns='geometry').to_csv(BASE/'processed'/'features_clean.csv',index=False)


## 8. Land suitability policy and sensitivity

Default policy for interpreting a shortlist: **exclude** clearly built-up/water-dominated cells from candidate screening, but retain them in the descriptive clustering dataset so their profiles remain visible. For sensitivity, compare thresholds rather than hiding the choice. You may change `BUILT_MAX` and `WATER_MAX`; cropland exclusion is a policy choice and should be tested explicitly.

In [ ]:
BUILT_MAX=20.0
WATER_MAX=10.0
CROP_MAX=None  # e.g. 50.0 only if justified; None means no cropland cutoff

def eligible_mask(df,built_max=BUILT_MAX,water_max=WATER_MAX,crop_max=CROP_MAX):
    ok=pd.Series(True,index=df.index)
    if 'pct_builtup' in df: ok &= df.pct_builtup.fillna(0)<=built_max
    if 'pct_water' in df: ok &= df.pct_water.fillna(0)<=water_max
    if crop_max is not None and 'pct_cropland' in df: ok &= df.pct_cropland.fillna(0)<=crop_max
    return ok

# Fallback to the full grid dataframe if model_df wasn't created due to single-feature constraints
target_df = model_df if model_df is not None else grid
if 'pct_builtup' not in target_df: print('Note: no land-cover layer yet, so these thresholds have no effect.')

for t in [10,20,30]:
    print('Built-up threshold',t,'% => eligible cells',int(eligible_mask(target_df,built_max=t).sum()),'/',len(target_df))

## 9. K-Means selection, stability, profiles

Use scaled features, compare inertia and silhouette across k=2…10, then judge interpretability. Silhouette can be costly on the full grid, so it uses a reproducible sample if needed. Labels are not ordinal; interpret only from the original-unit profile table.

In [ ]:
K_OVERRIDE=None   # set e.g. K_OVERRIDE=4 after reading the elbow/silhouette plots and the profiles
if model_df is not None and len(model_df)>=10:
    X0=model_df[FEATURES].astype(float)
    scaler=StandardScaler(); X=scaler.fit_transform(X0)
    ks=range(2,min(11,len(model_df)))
    results=[]
    for k in ks:
        km=KMeans(n_clusters=k,n_init=20,random_state=RNG).fit(X)
        ix=np.random.default_rng(RNG).choice(len(X),size=min(3000,len(X)),replace=False)
        sil=silhouette_score(X[ix],km.labels_[ix])
        results.append({'k':k,'inertia':km.inertia_,'silhouette':sil})
    scores=pd.DataFrame(results); display(scores)
    fig,ax=plt.subplots(1,2,figsize=(10,4)); ax[0].plot(scores.k,scores.inertia,'o-'); ax[0].set_title('Elbow'); ax[1].plot(scores.k,scores.silhouette,'o-'); ax[1].set_title('Silhouette'); plt.savefig(BASE/'figures'/'k_selection.png',dpi=200,bbox_inches='tight'); plt.show()
    K_CHOSEN=K_OVERRIDE or int(scores.sort_values('silhouette',ascending=False).iloc[0].k)  # silhouette is only a starting point
    print('Initial candidate k:',K_CHOSEN,'— inspect plots and profiles; change K_CHOSEN if needed.')
    km=KMeans(n_clusters=K_CHOSEN,n_init=50,random_state=RNG).fit(X)
    model_df['cluster']=km.labels_
    # Seed stability using ARI against the selected seed's labels.
    st=[]
    for seed in [1,7,21,42,84]:
        lab=KMeans(n_clusters=K_CHOSEN,n_init=20,random_state=seed).fit_predict(X)
        st.append({'seed':seed,'ARI_vs_seed42':adjusted_rand_score(model_df.cluster,lab)})
    display(pd.DataFrame(st))
    profiles=model_df.groupby('cluster')[FEATURES].mean().round(3)
    profiles.insert(0,'n_cells',model_df.groupby('cluster').size())
    display(profiles); profiles.to_csv(BASE/'processed'/'cluster_profiles.csv')
    model_df.drop(columns='geometry').to_csv(BASE/'processed'/'clustered_cells.csv',index=False)
    CLUSTER_DONE=True
else:
    print('Clustering skipped: Section 7 did not produce model_df (see its STOP message).')


## 10. Does the result depend on the model? Other clustering models

K-Means is one model with strong assumptions (round clusters of similar size). Here the **same scaled features** are given to five other models of different families:

| Model | Family | What is different from K-Means |
|---|---|---|
| Agglomerative (Ward) | hierarchical | merges cells bottom-up, no random start |
| Gaussian Mixture | probabilistic | clusters can be stretched and unequal; gives a probability per cell |
| Birch | tree-based | builds a summary tree first |
| Spectral | graph-based | follows chains of similar cells, clusters need not be round |
| HDBSCAN | density-based | chooses the number of clusters itself and may label cells as noise (-1) |

The first four are forced to the same `k` as K-Means so they are directly comparable. There is **no ground truth**, so agreement between models is evidence of robustness, not accuracy. Cluster numbers are re-aligned to the K-Means numbering so colours mean the same thing in every panel.

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.mixture import GaussianMixture
from sklearn.cluster import Birch, SpectralClustering
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

def align_to(ref, lab):
    # Cluster numbers are arbitrary. Rename each model's clusters so they match the K-Means numbering
    # as closely as possible (Hungarian matching); extra clusters get new numbers, noise stays -1.
    ref=np.asarray(ref); lab=np.asarray(lab)
    r_ids=np.unique(ref); l_ids=np.array([l for l in np.unique(lab) if l!=-1])
    ct=np.array([[np.sum((ref==r)&(lab==l)) for l in l_ids] for r in r_ids])
    ri,ci=linear_sum_assignment(-ct)
    mapping_={l_ids[c]:r_ids[r] for r,c in zip(ri,ci)}
    nxt=int(r_ids.max())+1
    for l in l_ids:
        if l not in mapping_: mapping_[l]=nxt; nxt+=1
    return np.array([mapping_.get(l,-1) for l in lab])

def solar_cluster(lab, pvout, min_share=0.02):
    # The cluster with the highest mean PVOUT, ignoring noise and tiny clusters (<2% of cells).
    s=pd.DataFrame({'c':np.asarray(lab),'p':np.asarray(pvout)}); s=s[s.c!=-1]
    g=s.groupby('c').p.agg(['mean','size']); g=g[g['size']>=min_share*len(lab)]
    return g['mean'].idxmax() if len(g) else None

MODELS_DONE=False
if CLUSTER_DONE:
    base=model_df['cluster'].to_numpy()
    SOLAR_COL=SOLAR_IN[0]
    MODEL_LABELS={'K-Means (baseline)':base}
    MODEL_LABELS['Agglomerative (Ward)']=AgglomerativeClustering(n_clusters=K_CHOSEN,linkage='ward').fit_predict(X)
    gmm=GaussianMixture(n_components=K_CHOSEN,covariance_type='full',n_init=5,random_state=RNG).fit(X)
    MODEL_LABELS['Gaussian Mixture']=gmm.predict(X)
    MODEL_LABELS['Birch']=Birch(n_clusters=K_CHOSEN).fit_predict(X)
    MODEL_LABELS['Spectral']=SpectralClustering(n_clusters=K_CHOSEN,affinity='nearest_neighbors',n_neighbors=15,
                                                 assign_labels='kmeans',random_state=RNG).fit_predict(X)
    try:
        from sklearn.cluster import HDBSCAN          # density-based: finds its own number of clusters, can label noise (-1)
        for mcs in (max(15,len(X)//50),15,8):         # try coarse first, then finer, until at least 2 dense groups appear
            hl=HDBSCAN(min_cluster_size=mcs).fit_predict(X)
            if len(set(hl)-{-1})>=2: MODEL_LABELS['HDBSCAN (density)']=hl; print('HDBSCAN min_cluster_size =',mcs); break
        else:
            print('HDBSCAN found no separate dense groups: the cells form one continuous cloud (a gradient), so it is left out of the table.')
    except ImportError:
        print('HDBSCAN needs scikit-learn >= 1.3; skipped.')
    MODEL_LABELS={n:(l if n.startswith('K-Means') else align_to(base,l)) for n,l in MODEL_LABELS.items()}

    base_solar=solar_cluster(base,model_df[SOLAR_COL]); base_set=set(np.where(base==base_solar)[0])
    rows=[]
    for name,lab in MODEL_LABELS.items():
        ok=lab!=-1; n_cl=len(np.unique(lab[ok]))
        sc=solar_cluster(lab,model_df[SOLAR_COL]); sset=set(np.where(lab==sc)[0]) if sc is not None else set()
        rows.append({'model':name,'n_clusters':n_cl,'noise_%':round(100*(~ok).mean(),1),
            'silhouette (higher=better)':silhouette_score(X[ok],lab[ok]) if n_cl>1 else np.nan,
            'davies_bouldin (lower=better)':davies_bouldin_score(X[ok],lab[ok]) if n_cl>1 else np.nan,
            'calinski_harabasz (higher=better)':calinski_harabasz_score(X[ok],lab[ok]) if n_cl>1 else np.nan,
            'ARI_vs_KMeans':adjusted_rand_score(base,lab),
            'solar_zone_overlap':len(base_set&sset)/len(base_set|sset) if (base_set|sset) else np.nan})
    MODEL_TABLE=pd.DataFrame(rows).set_index('model')
    display(MODEL_TABLE.round(3)); MODEL_TABLE.to_csv(BASE/'processed'/'model_comparison.csv')
    pd.DataFrame(MODEL_LABELS,index=model_df.cell_id).to_csv(BASE/'processed'/'model_labels.csv')
    print('ARI_vs_KMeans: 1 = identical grouping, 0 = no better than chance.')
    print(f'solar_zone_overlap: Jaccard overlap between each model\'s highest-{SOLAR_COL} cluster and the K-Means one (1 = same cells).')

    # Same clusters seen in 2-D (PCA) and on the map, one panel per model, colours matched to K-Means.
    pca=PCA(n_components=2,random_state=RNG); Z=pca.fit_transform(X)
    all_ids=sorted(set(np.concatenate(list(MODEL_LABELS.values())))-{-1})
    pal=dict(zip(all_ids,sns.color_palette('tab10',max(len(all_ids),3)))); pal[-1]=(.75,.75,.75)
    n=len(MODEL_LABELS); ncol=3; nrow=int(np.ceil(n/ncol))
    fig,axes=plt.subplots(nrow,ncol,figsize=(4.6*ncol,4*nrow)); axes=np.atleast_1d(axes).ravel()
    for ax,(name,lab) in zip(axes,MODEL_LABELS.items()):
        ax.scatter(Z[:,0],Z[:,1],c=[pal[l] for l in lab],s=6,alpha=.6); ax.set_title(name,fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[n:]: ax.set_axis_off()
    fig.suptitle(f'Clusters in PCA space (2 components explain {100*pca.explained_variance_ratio_.sum():.0f}% of variance)'); plt.tight_layout()
    plt.savefig(BASE/'figures'/'model_comparison_pca.png',dpi=200,bbox_inches='tight'); plt.show()

    geo=grid[['cell_id','geometry']].merge(pd.DataFrame(MODEL_LABELS,index=model_df.cell_id).reset_index(),on='cell_id',how='inner')
    fig,axes=plt.subplots(nrow,ncol,figsize=(4.2*ncol,5*nrow)); axes=np.atleast_1d(axes).ravel()
    for ax,name in zip(axes,MODEL_LABELS):
        geo.plot(ax=ax,color=[pal[l] for l in geo[name]],edgecolor='none'); country.boundary.plot(ax=ax,color='black',linewidth=.6)
        ax.set_title(name,fontsize=10); ax.set_axis_off()
    for ax in axes[n:]: ax.set_axis_off()
    handles=[plt.Line2D([],[],marker='s',ls='',color=pal[i],label=('noise' if i==-1 else f'cluster {i}')) for i in all_ids+([-1] if any((l==-1).any() for l in MODEL_LABELS.values()) else [])]
    fig.legend(handles=handles,loc='lower center',ncol=len(handles),frameon=False); plt.tight_layout(rect=(0,.03,1,1))
    plt.savefig(BASE/'figures'/'model_comparison_maps.png',dpi=200,bbox_inches='tight'); plt.show()
    MODELS_DONE=True
else:
    print('Model comparison skipped: K-Means (Section 9) did not run.')


### 10b. What the Gaussian Mixture adds

A probabilistic model answers two questions K-Means cannot: which `k` the data support under a likelihood criterion (BIC), and how confident the assignment of each cell is. Low-confidence cells are transition areas between zones and should not be over-interpreted.

In [ ]:
if MODELS_DONE:
    # 1) Which number of clusters does a probabilistic model prefer? (lowest BIC)
    bic=[]
    for k in range(2,11):
        g=GaussianMixture(n_components=k,covariance_type='full',n_init=3,random_state=RNG).fit(X)
        bic.append({'k':k,'BIC':g.bic(X)})
    bic=pd.DataFrame(bic); k_bic=int(bic.loc[bic.BIC.idxmin(),'k'])
    # 2) How sure is the model about each cell? K-Means cannot answer this.
    proba=gmm.predict_proba(X).max(axis=1)
    model_df['gmm_confidence']=proba
    unsure=float((proba<0.8).mean())
    fig,ax=plt.subplots(1,2,figsize=(11,4.5))
    ax[0].plot(bic.k,bic.BIC,'o-'); ax[0].axvline(K_CHOSEN,color='grey',ls='--',label=f'K-Means k={K_CHOSEN}'); ax[0].set_title('Gaussian Mixture BIC (lower is better)'); ax[0].set_xlabel('k'); ax[0].legend()
    conf=grid[['cell_id','geometry']].merge(model_df[['cell_id','gmm_confidence']],on='cell_id',how='inner')
    conf.plot(column='gmm_confidence',ax=ax[1],cmap='RdYlGn',vmin=.5,vmax=1,legend=True,edgecolor='none'); country.boundary.plot(ax=ax[1],color='black',linewidth=.6)
    ax[1].set_title('Assignment confidence (red = transition cells)'); ax[1].set_axis_off(); plt.tight_layout()
    plt.savefig(BASE/'figures'/'gmm_bic_confidence.png',dpi=200,bbox_inches='tight'); plt.show()
    print(f'BIC prefers k={k_bic}; K-Means used k={K_CHOSEN}. (BIC often prefers more, smaller clusters: that refines the zones, it does not contradict them.)')
    print(f'{100*unsure:.1f}% of cells have assignment confidence below 0.8: these are transition cells between zones.')


### 10c. Another *type* of model: supervised checks

Clustering is unsupervised. Three supervised models are used here to test and explain the zones, scored with **spatial cross-validation** (whole ~50 km blocks held out) and balanced accuracy:

- **A. Surrogate decision tree** on the clustering features: if a depth-3 tree reproduces the zones, they can be described by a few readable thresholds.
- **B. Random forest on held-out variables** (never shown to K-Means): if it predicts the zones well above chance, the zones reflect real geographic structure and are not an artefact of the chosen inputs.
- **C. Permutation importance**: which inputs drive the zones.

Caution: test A is partly circular (the labels were built from those same features), so a high score there shows the zones are *simple*, not that they are *true*. Test B is the independent one.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

SUPERVISED_DONE=False
if CLUSTER_DONE:
    y=model_df['cluster'].to_numpy()
    # Spatial cross-validation: neighbouring cells look alike, so a random split would flatter the score.
    # Cells are grouped in ~50 km blocks and a whole block is either in training or in test.
    blocks=(np.floor(model_df.lat/0.5).astype(int).astype(str)+'_'+np.floor(model_df.lon/0.5).astype(int).astype(str)).to_numpy()
    cv=GroupKFold(n_splits=5)
    def cv_score(model,cols):
        pred=cross_val_predict(model,model_df[cols].astype(float),y,groups=blocks,cv=cv)
        return balanced_accuracy_score(y,pred)
    chance=1/len(np.unique(y))

    # A) Surrogate decision tree on the SAME features: can the zones be written as a few simple rules?
    tree=DecisionTreeClassifier(max_depth=3,class_weight='balanced',random_state=RNG)
    acc_tree=cv_score(tree,FEATURES)
    tree.fit(model_df[FEATURES].astype(float),y)
    print(f'A) Depth-3 decision tree reproducing the K-Means zones: balanced accuracy {acc_tree:.2f} (chance = {chance:.2f})')
    print(export_text(tree,feature_names=FEATURES,decimals=2))

    # B) Random forest on variables that were NOT given to K-Means: are the zones real outside the clustering inputs?
    #    GHI/DNI are left out (near-copies of PVOUT) and so is pct_tree_shrub_grass (it is 100 minus the other land-cover shares).
    EXTERNAL=[c for c in ['TEMP','OPTA','dist_substation_km','pct_water'] if c in model_df and c not in FEATURES and model_df[c].notna().any()]
    acc_ext=np.nan
    if EXTERNAL:
        rf_ext=make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RNG,n_jobs=-1))
        acc_ext=cv_score(rf_ext,EXTERNAL)
        print(f'B) Random forest using only held-out variables {EXTERNAL}: balanced accuracy {acc_ext:.2f} (chance = {chance:.2f})')
    else:
        print('B) skipped: no held-out variables available.')

    # C) Which inputs drive the zones? Permutation importance of a random forest trained on the clustering features.
    rf=RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RNG,n_jobs=-1).fit(model_df[FEATURES].astype(float),y)
    imp=permutation_importance(rf,model_df[FEATURES].astype(float),y,scoring='balanced_accuracy',n_repeats=10,random_state=RNG)
    IMPORTANCE=pd.Series(imp.importances_mean,index=FEATURES).sort_values()
    IMPORTANCE.plot.barh(figsize=(7,3.5),color='#40566b'); plt.xlabel('Drop in balanced accuracy when the feature is shuffled'); plt.title('What drives the zones (permutation importance)')
    plt.tight_layout(); plt.savefig(BASE/'figures'/'feature_importance.png',dpi=200,bbox_inches='tight'); plt.show()
    pd.DataFrame({'test':['surrogate_tree_same_features','random_forest_held_out_features','chance_level'],
                  'balanced_accuracy':[acc_tree,acc_ext,chance]}).to_csv(BASE/'processed'/'supervised_validation.csv',index=False)
    SUPERVISED_DONE=True
else:
    print('Supervised check skipped: K-Means (Section 9) did not run.')


## 11. Maps, zone interpretation, and reporting

Create high-resolution cluster and input maps; use profiles to name zones only after examining measured values. Suggested candidates are those with high PVOUT, low built-up/water/cropland conflict, and reasonable infrastructure access—but never call them build-ready without grid capacity, ownership, environmental, legal, and field checks.

In [ ]:
if CLUSTER_DONE:
    mapdf=grid[['cell_id','geometry']].merge(model_df[['cell_id','cluster']],on='cell_id',how='left')
    ax=country.boundary.plot(figsize=(8,8),color='black',linewidth=1)
    mapdf.plot(column='cluster',ax=ax,categorical=True,legend=True,cmap='tab10',edgecolor='none',missing_kwds={'color':'lightgrey','label':'not clustered'})
    ax.set_title('Tunisia solar resource profile clusters'); ax.set_axis_off(); plt.tight_layout()
    plt.savefig(BASE/'figures'/'clusters_map.png',dpi=200,bbox_inches='tight'); plt.show()
else:
    print("Skipping cluster map because model_df or cluster labels were not generated.")

for col in ['PVOUT','pop_density','pct_bare','dist_power_line_km']:
    if col in grid and grid[col].notna().any():
        ax=grid.plot(column=col,figsize=(6,7),legend=True,cmap='viridis',missing_kwds={'color':'lightgrey'})
        country.boundary.plot(ax=ax,color='black',linewidth=.7); ax.set_title(col); ax.set_axis_off()
        plt.savefig(BASE/'figures'/f'{col}_map.png',dpi=200,bbox_inches='tight'); plt.show()
print('Saved outputs in',BASE/'processed','and',BASE/'figures')

## 12. Cluster-based screening shortlist (no hand-picked weights)

The earlier version ranked cells with a hand-set formula (40/25/20/15) that ignored the clusters. This version keeps K-Means central:

1. Read `cluster_profiles.csv` (Section 9) and decide which cluster(s) look like worthwhile candidate zones. Put their numbers in `CANDIDATE_CLUSTERS`.
2. Keep only **eligible** cells (Section 8 thresholds).
3. Rank the remaining cells on four criteria using **national percentile ranks**. Because any single set of weights is arbitrary, the ranking is repeated with 1,000 random weight vectors; `top_freq` is the share of those runs in which a cell lands in the top N. Cells with high `top_freq` are robust to the weighting choice.

This is a screening list for further study, not a siting decision.

In [ ]:
CANDIDATE_CLUSTERS=[1,2,3,4,5]   # e.g. [1,3], chosen AFTER reading cluster_profiles; empty means no shortlist yet
N_TOP=100

if not CLUSTER_DONE:
    print('[-] Cannot run shortlist: K-Means clustering (Section 9) was not completed.')
    print('    Please check Section 7 to ensure you have loaded at least 3 features (e.g. by enabling Earth Engine or OSM).')
elif not CANDIDATE_CLUSTERS:
    print('[!] Please inspect the cluster profiles in Section 9, then list your selected candidate cluster integers in CANDIDATE_CLUSTERS (e.g., CANDIDATE_CLUSTERS=[0, 2]).')
else:
    df=model_df.copy()
    df['eligible']=eligible_mask(df)
    absent=sorted(set(CANDIDATE_CLUSTERS)-set(df.cluster.unique()))
    if absent: print(f'NOTE: clusters {absent} do not exist (labels are {sorted(int(c) for c in df.cluster.unique())}); they are ignored. Edit CANDIDATE_CLUSTERS.')
    short=df[df.cluster.isin(CANDIDATE_CLUSTERS)&df.eligible].copy()
    print(f'{len(short)} eligible cells in clusters {CANDIDATE_CLUSTERS}')
    if len(short)==0:
        print('Nothing to rank: relax the eligibility thresholds or pick other clusters.')
    else:
        crit={'PVOUT':+1,'pct_bare':+1,'dist_power_line_km':-1,'pop_density':-1}   # +1 higher is better, -1 lower is better
        crit={k:v for k,v in crit.items() if k in df and df[k].notna().any()}
        pr=pd.DataFrame({k:(df[k].rank(pct=True) if v>0 else 1-df[k].rank(pct=True)) for k,v in crit.items()}).loc[short.index]
        rng=np.random.default_rng(RNG); hits=pd.Series(0.0,index=pr.index)
        for _ in range(1000):
            w=rng.dirichlet(np.ones(pr.shape[1]))
            hits[(pr@w).nlargest(N_TOP).index]+=1
        short['top_freq']=hits/1000
        short['equal_weight_score']=pr.mean(axis=1)*100
        cols=['cell_id','governorate','cluster','top_freq','equal_weight_score']+list(crit)
        top=short.sort_values(['top_freq','equal_weight_score'],ascending=False)[cols].head(N_TOP)
        print('Criteria used:',crit)
        display(top.round(2).reset_index(drop=True))
        short[cols].sort_values('top_freq',ascending=False).to_csv(BASE/'processed'/'shortlist.csv',index=False)

        plt.figure(figsize=(9,6))
        sns.scatterplot(data=df,x='pop_density',y='PVOUT',hue='cluster',palette='Set2',alpha=.5,s=18)
        top_xy=df.set_index('cell_id').loc[top.cell_id,['pop_density','PVOUT']]
        plt.scatter(top_xy.pop_density,top_xy.PVOUT,s=120,facecolors='none',edgecolors='black',linewidths=1.4,label=f'top {N_TOP} shortlist')
        plt.xscale('symlog',linthresh=1); plt.xlabel('Population density (people/km2, symlog)'); plt.ylabel('PVOUT')
        plt.title('Solar potential vs population, by cluster'); plt.legend(bbox_to_anchor=(1.02,1),loc='upper left'); plt.tight_layout()
        plt.savefig(BASE/'figures'/'shortlist_scatter.png',dpi=200,bbox_inches='tight'); plt.show()


### 12b. Is the shortlist robust to the choice of model?

For each model from Section 10, the share of shortlisted cells that fall inside that model's highest-solar cluster. If the share stays high, the shortlist does not depend on having used K-Means. A short summary of all the model checks follows.

In [ ]:
if CLUSTER_DONE and globals().get('MODELS_DONE') and 'top' in globals():
    # Does the final answer (the shortlist) survive a change of model?
    pos=pd.Series(np.arange(len(model_df)),index=model_df.cell_id).loc[top.cell_id].to_numpy()
    rob=[]
    for name,lab in MODEL_LABELS.items():
        sc=solar_cluster(lab,model_df[SOLAR_COL])
        rob.append({'model':name,'shortlist_in_solar_zone_%':round(100*float((lab[pos]==sc).mean()),1) if sc is not None else np.nan})
    rob=pd.DataFrame(rob).set_index('model'); display(rob)
    rob.to_csv(BASE/'processed'/'shortlist_model_robustness.csv')

    def grade(v,hi,lo): return 'strong' if v>=hi else ('moderate' if v>=lo else 'weak')
    others=MODEL_TABLE.drop(index='K-Means (baseline)')
    part=others[others.index!='HDBSCAN (density)']      # models forced to the same k: directly comparable
    print('SUMMARY (read together with the tables and maps above)')
    print(f"- Agreement of same-k models with K-Means (ARI): min {part.ARI_vs_KMeans.min():.2f}, mean {part.ARI_vs_KMeans.mean():.2f} -> {grade(part.ARI_vs_KMeans.mean(),.8,.5)}")
    print(f"- High-solar zone overlap across all models (Jaccard): min {others.solar_zone_overlap.min():.2f}, mean {others.solar_zone_overlap.mean():.2f} -> {grade(others.solar_zone_overlap.mean(),.8,.5)}")
    print(f"- Shortlist cells inside each model's high-solar zone: min {rob.iloc[:,0].min():.0f}%, mean {rob.iloc[:,0].mean():.0f}% -> {grade(rob.iloc[:,0].mean(),90,70)}")
    if globals().get('SUPERVISED_DONE'):
        print(f"- Zones as simple rules (depth-3 tree, spatial CV): {acc_tree:.2f} vs chance {chance:.2f}")
        if not np.isnan(acc_ext): print(f"- Zones predicted from held-out variables (random forest, spatial CV): {acc_ext:.2f} vs chance {chance:.2f}")
    best=MODEL_TABLE['silhouette (higher=better)'].idxmax()
    print(f"- Best silhouette: {best}. These scores describe cluster shape only; there is no ground truth, so no model is 'the correct one'.")
else:
    print('Robustness summary skipped: run Sections 9, 10 and 12 first.')


In [ ]:
import matplotlib.pyplot as plt

# Filter the grid geometries to only include the shortlisted cells
highlight_gdf = grid[grid['cell_id'].isin(top['cell_id'])]

fig, ax = plt.subplots(figsize=(10, 10))

# Plot the background governorates
governorates.plot(ax=ax, color='#f3f4f6', edgecolor='#9ca3af', linewidth=0.5, label='Governorates')

# Plot the empty grid lines with high transparency
grid.boundary.plot(ax=ax, color='#374151', linewidth=0.1, alpha=0.5, label='Grid')

# Highlight the top shortlisted cells in red
highlight_gdf.plot(ax=ax, color='#ef4444', edgecolor='#b91c1c', linewidth=1.5, alpha=0.85, label='Top Shortlisted Zones')

# Draw the national boundary on top
country.boundary.plot(ax=ax, color='#111827', linewidth=1.2)

# Plot configuration
plt.title(f'Tunisia Map: Location of Top {len(top)} Screened Solar Priority Zones', fontsize=14, fontweight='bold', pad=15)
plt.axis('off')
plt.tight_layout()

# Save figure
plt.savefig(BASE/'figures'/'highlighted_shortlist_map.png', dpi=200, bbox_inches='tight')
plt.show()

# Display the governorates where the top cells are located
print("Top cells are located in the following governorates:")
print(highlight_gdf[['cell_id', 'governorate']].to_string(index=False))

In [ ]:
import folium

# Filter the grid geometries to only include our top shortlisted cells and project to WGS84 (EPSG:4326) for Leaflet mapping
highlight_wgs84 = grid[grid['cell_id'].isin(top['cell_id'])].to_crs(epsg=4326)

# Initialize map centered on southern Tunisia
m = folium.Map(
    location=[31.5, 9.5],
    zoom_start=7,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community'
)

# Add the shortlisted grid cells as an overlay
folium.GeoJson(
    highlight_wgs84,
    name='Top Shortlisted Zones',
    style_function=lambda feature: {
        'fillColor': '#ef4444',
        'color': '#b91c1c',
        'weight': 2,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(fields=['cell_id', 'governorate', 'PVOUT', 'pop_density'], aliases=['Cell ID:', 'Governorate:', 'PVOUT:', 'Pop Density:'])
).add_to(m)

# Add layer control to toggle overlays
folium.LayerControl().add_to(m)

# Display the interactive map
m

In [ ]:
import folium

# Project geometries to WGS84 (EPSG:4326) for Leaflet
highlight_wgs84 = grid[grid['cell_id'].isin(top['cell_id'])].to_crs(epsg=4326)
roads_wgs84 = roads.to_crs(epsg=4326)
lines_wgs84 = lines.to_crs(epsg=4326)
subs_wgs84 = subs.to_crs(epsg=4326)

# Initialize map centered on Tunisia
m2 = folium.Map(
    location=[34.0, 9.5],
    zoom_start=7,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community'
)

# Street basemap from Esri: same server as the satellite layer above, no API key needed.
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri, HERE, Garmin, USGS, NGA, and the GIS User Community',
    name='Street map (Esri)'
).add_to(m2)

# 1. Overlay: Top Shortlisted Zones (Grid Cells)
folium.GeoJson(
    highlight_wgs84,
    name=f'Top {len(top)} Shortlisted Zones',
    style_function=lambda feature: {
        'fillColor': '#ef4444',
        'color': '#b91c1c',
        'weight': 2,
        'fillOpacity': 0.45
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['cell_id', 'governorate', 'PVOUT', 'pop_density'],
        aliases=['Cell ID:', 'Governorate:', 'PVOUT:', 'Pop Density:']
    )
).add_to(m2)

# 2. Overlay: Major Roads
folium.GeoJson(
    roads_wgs84,
    name='Major Roads (OSM)',
    style_function=lambda feature: {
        'color': '#f59e0b',
        'weight': 1.5,
        'opacity': 0.8
    }
).add_to(m2)

# 3. Overlay: Power Lines
folium.GeoJson(
    lines_wgs84,
    name='Power Transmission Lines (OSM)',
    style_function=lambda feature: {
        'color': '#3b82f6',
        'weight': 2.0,
        'opacity': 0.9,
        'dashArray': '5, 5'
    }
).add_to(m2)

# 4. Overlay: Substations & Plants
for _, row in subs_wgs84.iterrows():
    geom = row.geometry
    if geom.geom_type == 'Point':
        folium.CircleMarker(
            location=[geom.y, geom.x],
            radius=5,
            color='#10b981',
            fill=True,
            fill_color='#059669',
            fill_opacity=0.9,
            tooltip='Substation/Power Plant'
        ).add_to(m2)
    else:
        # For polygons/multipolygons, represent as centroids or standard polygons
        centroid = geom.centroid
        folium.CircleMarker(
            location=[centroid.y, centroid.x],
            radius=5,
            color='#10b981',
            fill=True,
            fill_color='#059669',
            fill_opacity=0.9,
            tooltip='Substation/Power Plant'
        ).add_to(m2)

# Add Layer Control to easily toggle these overlays
folium.LayerControl(collapsed=False).add_to(m2)

# Display the map
m2

### Report notes to complete after running

- **Data:** GeoBoundaries ADM0/ADM1; Global Solar Atlas (record layers, units, daily/yearly convention); WorldPop 2020; ESA WorldCover 2021; OpenStreetMap; NASA POWER only if used.
- **Method:** UTM zone 32N; 10 km nominal cells; zonal means for continuous resource, sums for population; logged/skewed features and standard scaling; chosen k with elbow/silhouette rationale; land eligibility thresholds.
- **Results:** Cite the profile table and cluster map, describe meaningful trade-offs, include seed stability, the model-comparison table (Section 10), the held-out-variable score (Section 10c) and the shortlist robustness table (Section 12b).
- **Limitations:** Long-term solar averages omit slope/shading/ownership; source years differ; OSM can be incomplete and grid capacity is absent; K-Means is scale/k sensitive; spatial autocorrelation means cells are not independent. These clusters are screening profiles, not recommendations to build.

**Credits:** Global Solar Atlas (World Bank Group, ESMAP, Solargis), CC BY 4.0; WorldPop, University of Southampton; ESA WorldCover 10 m 2021 v200 (Zanaga et al. 2022, doi:10.5281/zenodo.7254221); © OpenStreetMap contributors (ODbL); geoBoundaries (Runfola et al. 2020, PLOS ONE); NASA POWER acknowledgement as specified by NASA POWER.